***Network Anomaly Detection System***

### **Problem Statement:**

The objective of this project is to build and compare supervised and unsupervised machine learning models for detecting anomalous network traffic under a realistic class imbalance scenario (~5% attacks). The system aims to accurately identify rare intrusion attempts while minimizing false positives, and to analyze the performance trade-offs between classification-based and anomaly detection approaches.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder,StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.decomposition import PCA

from sklearn.ensemble import IsolationForest, RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.neighbors import LocalOutlierFactor as LOF

from sklearn.metrics import (
    accuracy_score, classification_report,
    precision_score, recall_score,
    f1_score, roc_auc_score,
    average_precision_score
)

from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam

In [ ]:
#Load the dataset
df_train=pd.read_csv('/content/Train_data.csv.zip')

In [ ]:
df_train.shape

In [ ]:
df_train.info()

In [ ]:
#check class distribution
df_train['class'].value_counts()

In [ ]:
#creating unbalanced dataset with 5% anomalies
df_normal = df_train[df_train['class'] == 'normal']
df_anomaly = df_train[df_train['class'] == 'anomaly']

df_anomaly_small = df_anomaly.sample(
    n=int(len(df_normal) * 0.05),
    random_state=42
)

df_imbalanced = pd.concat([df_normal, df_anomaly_small])

df_imbalanced['class'].value_counts()

In [ ]:
#shuffle after combining
df_imbalanced = df_imbalanced.sample(frac=1, random_state=42).reset_index(drop=True)

In [ ]:
df_imbalanced.shape

In [ ]:
X = df_imbalanced.drop("class", axis=1)
y = df_imbalanced["class"]

In [ ]:
#training and validation split
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
X_train.shape, y_train.shape, X_val.shape, y_val.shape

In [ ]:
#Onehotencoding
combined = pd.concat([X_train, X_val])
combined = pd.get_dummies(combined)

X_train = combined.iloc[:len(X_train)]
X_val = combined.iloc[len(X_train):]

In [ ]:
#Manually encoding target variable
y_train.unique()
y_train=y_train.map({'normal':0,'anomaly':1})
y_val=y_val.map({'normal':0,'anomaly':1})

In [ ]:
X_train.shape,X_val.shape

In [ ]:
#Scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

In [ ]:
#SMOTE: oversampling technique that generates synthetic minority class samples
#to address class imbalance
#SMOTE applied only on training data

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train_scaled, y_train)

Pipeline 1 (Supervised):

Split
→ Encode
→ Scale
→ SMOTE
→ RF / XGB

*RandomForest Classifier*

In [ ]:
rf=RandomForestClassifier(n_estimators=100,random_state=42)
rf.fit(X_train_sm,y_train_sm)

In [ ]:
rf_pred = rf.predict(X_val_scaled)
rf_prob = rf.predict_proba(X_val_scaled)[:, 1]

#evaluation
accuracy = accuracy_score(y_val, rf_pred)
print(classification_report(y_val, rf_pred))

*XGBooster Classifier*

In [ ]:
xgb=XGBClassifier()
xgb.fit(X_train_sm,y_train_sm)
xgb_pred = xgb.predict(X_val_scaled)
xgb_prob = xgb.predict_proba(X_val_scaled)[:, 1]

In [ ]:
#evaluation
accuracy = accuracy_score(y_val, xgb_pred)
print(classification_report(y_val,xgb_pred))

Pipeline 2(Unsupervised):

Split
→ Encode
→ Scale
→ PCA (only for LOF)
→ Select normal samples
→ Fit IF/LOF/AE
→ Predict on full validation set

*Isolation Forest*

In [ ]:
X_train_normal = X_train_scaled[y_train == 0] # only on training set

isolation_forest = IsolationForest(n_estimators=200,contamination=0.05, random_state=42)
isolation_forest.fit(X_train_normal)

In [ ]:
# Anomaly scores (higher = more anomalous)
if_scores = -isolation_forest.decision_function(X_val_scaled)

# Binary predictions
if_pred = isolation_forest.predict(X_val_scaled)
if_pred = np.where(if_pred == -1, 1, 0)

# Average precision
avg_prec = average_precision_score(y_val, if_scores)

print("Average Precision:", avg_prec)
print(classification_report(y_val, if_pred))

*Local Outlier Factor*


In [ ]:
#reduce dimensions for LOF
pca = PCA(n_components=0.95)
# Keep enough principal components to preserve 95% of the total variance in the data.
X_train_pca = pca.fit_transform(X_train_scaled)
X_val_pca = pca.transform(X_val_scaled)

In [ ]:
X_train_normal = X_train_pca[y_train == 0] # only on training set

lof=LOF(n_neighbors=20,contamination=0.05,novelty=True)
lof.fit(X_train_normal)

In [ ]:
lof_pred = lof.predict(X_val_pca)
lof_pred = np.where(lof_pred == -1, 1, 0)

# Anomaly scores (higher = more anomalous)
lof_scores = -lof.decision_function(X_val_pca)

# Average precision
avg_prec = average_precision_score(y_val, lof_scores)

print("Average Precision:", avg_prec)
print(classification_report(y_val, lof_pred))

*Autoencoder*


In [ ]:
X_train_normal = X_train_scaled[y_train == 0] # only on training set

input_dim = X_train_normal.shape[1]

input_layer = Input(shape=(input_dim,))
encoded = Dense(64, activation="relu")(input_layer)
encoded = Dense(32, activation="relu")(encoded)
decoded = Dense(64, activation="relu")(encoded)
decoded = Dense(input_dim, activation="linear")(decoded)

autoencoder = Model(inputs=input_layer, outputs=decoded)
autoencoder.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mse"
)

autoencoder.fit(
    X_train_normal,
    X_train_normal,
    epochs=20,
    batch_size=256,
    validation_split=0.2,
    shuffle=True
)


reconstructions = autoencoder.predict(X_val_scaled)
mse = np.mean(np.power(X_val_scaled - reconstructions, 2), axis=1) #anomaly scores

threshold = np.percentile(mse, 95)

ae_pred = (mse > threshold).astype(int)

print(classification_report(y_val, ae_pred))

In [ ]:
def evaluate_model(name, y_true, y_pred, scores):
    return {
        "Model": name,
        "Precision": precision_score(y_true, y_pred),
        "Recall": recall_score(y_true, y_pred),
        "F1": f1_score(y_true, y_pred),
        "ROC-AUC": roc_auc_score(y_true, scores),
        "PR-AUC": average_precision_score(y_true, scores)
    }

In [ ]:
results = []

results.append(evaluate_model("RandomForest", y_val, rf_pred, rf_prob))
results.append(evaluate_model("XGBoost", y_val, xgb_pred, xgb_prob))
results.append(evaluate_model("IsolationForest", y_val, if_pred, if_scores))
results.append(evaluate_model("LOF", y_val, lof_pred, lof_scores))
results.append(evaluate_model("Autoencoder", y_val, ae_pred, mse))

results_df = pd.DataFrame(results)
results_df.sort_values(by="PR-AUC", ascending=False)

# Conclusion — Network Intrusion Detection Project

This project compared supervised and unsupervised approaches for detecting anomalous network traffic under a realistic 95–5 class imbalance scenario.


Supervised models significantly outperformed unsupervised methods, achieving near-perfect PR-AUC scores due to their ability to learn explicit decision boundaries from labeled data. Among unsupervised models, IsolationForest showed the strongest performance, while LOF was sensitive to high-dimensional data despite PCA-based reduction.

Overall, the results demonstrate that supervised models are best when labeled attack data is available, while unsupervised methods remain valuable for detecting unseen or zero-day anomalies.